In [5]:
import numpy as np
import pandas as pd

# Load real BTCUSD 1-min data and compute simple returns from Close price
csv_path = "./data/btcusd_1-min_data.csv"
org_prices = pd.read_csv(csv_path)["Close"].astype(float).values

In [6]:
# Analyze returns and compute 1% / 99% quantiles

import numpy as np

# Assuming `prices` and `returns` are defined in the previous cell
org_returns = org_prices[1:] / org_prices[:-1] - 1

# Basic stats on original prices
print(f"Price sample size: {len(org_returns)}")
print(f"Price mean: {org_returns.mean():.4f}, std: {org_returns.std(ddof=1):.4f}")

# # Signal thresholds
lower_quantile = 0.1
upper_quantile = 0.9
sell_threshold = np.quantile(org_returns, lower_quantile)
buy_threshold = np.quantile(org_returns, upper_quantile)

print("\nReturn quantiles:")
print(f"{lower_quantile} % quantile: {sell_threshold:.6f} ({sell_threshold:.4%})")
print(f"{upper_quantile} % quantile: {buy_threshold:.6f} ({buy_threshold:.4%})")

Price sample size: 7460317
Price mean: 0.0000, std: 0.0018

Return quantiles:
0.1 % quantile: -0.000977 (-0.0977%)
0.9 % quantile: 0.000987 (0.0987%)


In [7]:
# XGBoost: 3-class direction from past 1-min returns (research notebook)
# Production hourly harness: strategy.py + uv run backtest.py
# Uses buy_threshold / sell_threshold from the quantile cell above.

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


def label_direction(x):
    if x > buy_threshold:
        return "up"
    elif x < sell_threshold:
        return "down"
    else:
        return "flat"


window_size = 30
xgb_test_size = 0.3

# Last N minutes only (full org_prices is millions of rows — increase if you want)
prices = org_prices[-60 * 24 * 7 :]
returns = prices[1:] / prices[:-1] - 1

X, y = [], []
for t in range(window_size, len(returns) - 1):
    X.append(returns[t - window_size : t])
    y.append(label_direction(returns[t]))

X = np.array(X)
y = np.array(y)

class_map = {"up": 0, "flat": 1, "down": 2}
y_int = np.array([class_map[c] for c in y])

print("X shape:", X.shape)
print("Class distribution:")
for cls in ["up", "flat", "down"]:
    print(f"  {cls}: {(y == cls).sum()}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y_int, test_size=xgb_test_size, shuffle=False
)

model = None
model_backend = None
try:
    import xgboost as xgb

    model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        n_jobs=-1,
        random_state=42,
    )
    model.fit(X_train, y_train)
    model_backend = "xgboost"
except Exception as e:
    print("XGBoost unavailable:", e)
    print("Using sklearn HistGradientBoostingClassifier instead.")
    from sklearn.ensemble import HistGradientBoostingClassifier

    model = HistGradientBoostingClassifier(
        max_iter=200,
        max_depth=3,
        learning_rate=0.1,
        random_state=42,
    )
    model.fit(X_train, y_train)
    model_backend = "sklearn HistGradientBoostingClassifier"

y_pred_int = model.predict(X_test)
print(f"\nBackend: {model_backend}")
print(f"Hold-out accuracy: {accuracy_score(y_test, y_pred_int):.4f}")


X shape: (10048, 30)
Class distribution:
  up: 751
  flat: 8526
  down: 771

Backend: xgboost
Hold-out accuracy: 0.8733


In [8]:
# Backtest predicted signals (same style as ARIMA backtest metrics)

import numpy as np

transaction_cost = 0.0
pred_to_signal = {0: "buy", 1: "hold", 2: "sell"}

n_train = len(X_train)
y_pred_test = model.predict(X_test)

k_test = np.arange(n_train, n_train + len(X_test), dtype=int)
t_idx = window_size + k_test
trade_returns = returns[t_idx]

signals_arr = np.array([pred_to_signal[int(p)] for p in y_pred_test])

unique_sigs, sig_counts = np.unique(signals_arr, return_counts=True)
print("Signal totals (from predictions):")
for s, c in zip(unique_sigs, sig_counts):
    print(f"  {s}: {c}")

positions = np.zeros(len(signals_arr), dtype=float)
for i, sig in enumerate(signals_arr):
    if sig == "buy":
        positions[i] = 1.0
    elif sig == "sell":
        positions[i] = -1.0
    else:
        positions[i] = 0.0

if len(trade_returns) != len(positions):
    raise ValueError("Length mismatch between positions and trade_returns.")

gross_strategy_ret = positions * trade_returns
position_changes = np.abs(np.diff(positions, prepend=0.0))
net_strategy_ret = gross_strategy_ret - transaction_cost * position_changes

k_train = np.arange(0, n_train, dtype=int)
train_returns_aligned = returns[window_size + k_train]
fallback_train = float(np.mean(train_returns_aligned))
mean_ret_by_class = {}
for c in (0, 1, 2):
    mask = y_train == c
    mean_ret_by_class[c] = (
        train_returns_aligned[mask].mean() if np.any(mask) else fallback_train
    )

preds_arr = np.array([mean_ret_by_class[int(p)] for p in y_pred_test], dtype=float)
realized_arr = trade_returns

mse = np.mean((preds_arr - realized_arr) ** 2)
mae = np.mean(np.abs(preds_arr - realized_arr))
nz = realized_arr != 0
mape = (
    np.mean(np.abs((realized_arr[nz] - preds_arr[nz]) / realized_arr[nz]))
    if np.any(nz)
    else np.nan
)

pred_dirs = np.array([label_direction(x) for x in preds_arr])
real_dirs = np.array([label_direction(x) for x in realized_arr])

labels = ["up", "flat", "down"]
conf_mat = np.zeros((3, 3), dtype=int)
for r, f in zip(real_dirs, pred_dirs):
    conf_mat[labels.index(r), labels.index(f)] += 1

cum_return = (1 + net_strategy_ret).prod() - 1
mean_ret = net_strategy_ret.mean()
vol = net_strategy_ret.std(ddof=1)
sharpe = mean_ret / vol * np.sqrt(60) if vol > 0 else np.nan

print("\nDirectional confusion matrix (rows = actual, cols = predicted):")
print("            up   flat  down")
for i, lab in enumerate(labels):
    row = conf_mat[i]
    print(f"{lab:>5}  {row[0]:6d} {row[1]:6d} {row[2]:6d}")

print("\nF1 scores by direction:")
for i, lab in enumerate(labels):
    tp = conf_mat[i, i]
    fp = conf_mat[:, i].sum() - tp
    fn = conf_mat[i, :].sum() - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else np.nan
    )
    print(f"{lab:>5}: precision={precision:.3f}, recall={recall:.3f}, F1={f1:.3f}")

print()
print(f"Model: {model_backend}")
print(
    f"Train/test split: {1 - xgb_test_size:.0%} / {xgb_test_size:.0%} (chronological); metrics on test only"
)
print(f"Window size: {window_size} minutes")
print(f"Transaction cost per position change: {transaction_cost}")
print(f"Number of non-zero positions: {np.count_nonzero(positions)}")
print(f"Cumulative return: {cum_return:.2%}")
print(f"Average per-minute return: {mean_ret:.4%}")
print(f"Per-minute volatility: {vol:.4%}")
print(f"Sharpe ratio (per hour, ~60 min): {sharpe:.2f}")
print(f"MSE of return forecasts: {mse:.6e}")
print(f"MAE of return forecasts: {mae:.6e}")
print(f"MAPE of return forecasts: {mape:.4%}")



Signal totals (from predictions):
  buy: 6
  hold: 2997
  sell: 12

Directional confusion matrix (rows = actual, cols = predicted):
            up   flat  down
   up       2    189      1
 flat       3   2625      5
 down       1    183      6

F1 scores by direction:
   up: precision=0.333, recall=0.010, F1=0.020
 flat: precision=0.876, recall=0.997, F1=0.933
 down: precision=0.500, recall=0.032, F1=0.059

Model: xgboost
Train/test split: 70% / 30% (chronological); metrics on test only
Window size: 30 minutes
Transaction cost per position change: 0.0
Number of non-zero positions: 18
Cumulative return: 1.07%
Average per-minute return: 0.0004%
Per-minute volatility: 0.0105%
Sharpe ratio (per hour, ~60 min): 0.26
MSE of return forecasts: 5.749957e-07
MAE of return forecasts: 5.002334e-04
MAPE of return forecasts: 106.3887%
